In [1]:
import math
from typing import Callable
from statistics import NormalDist

import numpy as np

def prom_update(prom_prev: float, x_new: float, n_curr: int):
    if n_curr < 1:
        raise Exception()
    elif n_curr == 1:
        return x_new
    else:
        return prom_prev + (x_new - prom_prev) / n_curr

def calculo_z_standard(alpha: float):
    return NormalDist().inv_cdf(1 - alpha / 2)

def intervalo(prom: float, s_cuad: float, n: int, alpha: float):
    z_alpha_2 = calculo_z_standard(alpha)

    std = math.sqrt(s_cuad/n)
    izq = prom - z_alpha_2 * std
    der = prom + z_alpha_2 * std

    return izq, der

### Ejercicio 4.
Estimar $\pi$ sorteando puntos uniformemente distribuidos en el cuadrado cuyos vértices son:
$(1, 1), (−1, 1), (−1, −1), (1, −1)$, y contabilizando la fracción que cae dentro del círculo inscripto de radio 1.

a. Dar la estimación de la proporción de puntos que caen dentro del círculo deteniendo la simulación cuando la desviación estándar muestral del estimador sea menor que $0.01$.

b. Construir un intervalo de confianza del $95\%$ para $\pi$ cuyo ancho sea menor que (a) $0.1$, (b) $0.05$, (c) $0.001$. ¿Cuántas simulaciones fueron necesarias en cada caso?

---

#### Análisis punto a
$$p = \frac{\text{area circulo}}{\text{area rectangulo}} = \frac{\pi r^2}{l^2}$$

Como es un cuadrado de 2*2 y el radio es 1, entonces
$$p = \frac{\pi}{2^2} = \frac{\pi}{4}$$

Tenemos que $4p = \pi$, por lo que podemos estimar $\hat{p}$ y con este estimar $\hat{\pi}$


Definimos entonces la variable aleatoria:
$$
X_i = \begin{cases}
    1 , & \text{si el punto cae dentro del círculo} \\
    0 , & \text{en otro caso}
\end{cases}
$$

Entonces:
$$
X_i \sim Bernoulli(p), \text{ por lo que}: \begin{cases}
    E[X_i] & = p \\
    Var(X_i) & = p(1-p)
\end{cases}
$$

Sea:

$$
\overline{X}_n=\frac{1}{n}\sum_{i=1}^{n}X_i
$$

Como $E[\overline{X}_n]=p$, un estimador para $\pi$ es:

$$
\hat{\pi}_n = 4\overline{X}_n
$$

Además:

$$
Var(\overline{X}_n) = \frac{p(1-p)}{n}
$$

Estimamos entonces su desvío estándar mediante:

$$
\sqrt{
\frac{\overline{X}_n(1-\overline{X}_n)}{n}
}
$$

In [2]:
def fun(u1: float, u2: float):
    x = u1 * 2 - 1 # x existe [-1, 1]
    y = u2 * 2 - 1 # y existe [-1, 1]

    return x**2 + y**2 <= 1

def punto_a():
    p = fun(np.random.random(), np.random.random())
    n = 1

    while n <= 100 or math.sqrt(p*(1 - p) / n) >= 0.01:
        X = fun(np.random.random(), np.random.random())
        n += 1

        p = prom_update(p, X, n)

    print("Punto: ", fun.__name__)
    print("Cantidad de datos generados: ", n)
    print(f"p = pi/4 ~ {p:.4f}")
    print(f"Desviación estándar muestral = {math.sqrt(p*(1 - p) / n):.4f} ")
    print(f"PI = {math.pi:.4f} ~ {p*4:.4f}")


punto_a()

Punto:  fun
Cantidad de datos generados:  1683
p = pi/4 ~ 0.7861
Desviación estándar muestral = 0.0100 
PI = 3.1416 ~ 3.1444


#### Análisis punto b
Como $p = \pi/4$ y sabemos que $\hat{p}_n = \overline{X}_n = \frac{1}{n}\sum_{i=1}^{n}X_i$.

Entonces tomo como estimador de $\hat{\pi}_n = 4\hat{p}_n = 4\overline{X}_n$.

Como queremos construir un intervalo de confianza del $95\%$ para $\pi$, tenemos que $I.C. = 0.95 = 1 - \alpha \implies \alpha = 0.05$, lo cual nos daría la siguiente aprox.
$$
\hat{\pi}_n \pm z_{\frac{\alpha}{2}} \sqrt{ Var(\hat{\pi}_n) }.
$$

La amplitud del intervalo es:
$$
L = 2 \cdot z_{\frac{\alpha}{2}} \cdot \sqrt{ Var(\hat{\pi}_n) }
$$

Como sabemos que $Var(\hat{\pi}_n) = Var(4\hat{p}_n) = Var(4\overline{X}_n) =  4^2 Var(\overline{X}_n) = 16 \cdot \frac{\overline{X}_n(1-\overline{X}_n)}{n}$.

Por lo que:
$$


\begin{array}{cl}
        L & = 2 \cdot z_{\frac{\alpha}{2}} \cdot \sqrt{ Var(\hat{\pi}_n) } \\ \\

        \frac{L}{2 \cdot z_{\frac{\alpha}{2}}} & = \sqrt{ 16 \cdot \frac{\overline{X}_n(1-\overline{X}_n)}{n} } \\ \\

        \frac{L}{2 \cdot z_{\frac{\alpha}{2}}} & = 4 \cdot \sqrt{ \frac{\overline{X}_n(1-\overline{X}_n)}{n} } \\ \\

        \frac{L}{8 \cdot z_{\frac{\alpha}{2}}} & = \sqrt{ \frac{\overline{X}_n(1-\overline{X}_n)}{n} }
\end{array}
$$

Lo cual implica que la cota para saber cuantos puntos generar es $n \ge 100 \ \vee \ \frac{L}{8 \cdot z_{\frac{\alpha}{2}}} \le \sqrt{ \frac{\overline{X}_n(1-\overline{X}_n)}{n} }$

In [3]:
def simulaciones(fun: Callable[[float, float], float], l: float, alpha: float, n_min: int = 100):
    z_alpha_2 = calculo_z_standard(alpha)
    cota = l / (z_alpha_2 * 8)

    p = fun(np.random.random(), np.random.random())
    n = 1

    while n <= n_min or math.sqrt(p*(1 - p) / n) >= cota:
        X = fun(np.random.random(), np.random.random())
        n += 1

        p = prom_update(p, X, n)

    i_d, i_i = intervalo(p, math.sqrt(p*(1 - p) / n), n, alpha)

    print(
        f"{l:>9} | {n:9d} | {f"[{i_d:.4f}, {i_i:.4f}]":>18} | {math.sqrt(p*(1 - p) / n):10.4f} | {p*4:10.4f}")

alpha = 0.05
n_min = 100
print(f"{"Amplitud":>9} | {"N° Sim":>9} | {"Intervalo obtenido":>18} | {"Var":>10} | {"Aprox. PI":>10}")
simulaciones(fun, 0.1, alpha, n_min)
simulaciones(fun, 0.05, alpha, n_min)
simulaciones(fun, 0.001, alpha, n_min)

 Amplitud |    N° Sim | Intervalo obtenido |        Var |  Aprox. PI
      0.1 |      4171 |   [0.7811, 0.7859] |     0.0064 |     3.1340
     0.05 |     16691 |   [0.7825, 0.7842] |     0.0032 |     3.1334
    0.001 |  41445808 |   [0.7853, 0.7853] |     0.0001 |     3.1414
